In [ ]:
# prerequisites
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions.normal import Normal
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np

In [ ]:
class Gaussian2():
    def __init__(self, sigma, mu1=[-3, -3], mu2=[3, 3], pi1=0.1):
        self.x1_dist = Normal(mu1[0], sigma)
        self.y1_dist = Normal(mu1[1], sigma)
        self.x2_dist = Normal(mu2[0], sigma)
        self.y2_dist = Normal(mu2[1], sigma)
        self.pi1 = pi1
        self.pi2 = 1 - pi1

    def log_prob(self, data):
        x = data[:, 0]
        y = data[:, 1]
        log_prob1 = self.x1_dist.log_prob(x) + self.y1_dist.log_prob(y)
        log_prob2 = self.x2_dist.log_prob(x) + self.y2_dist.log_prob(y)
        return torch.log(self.pi1 * torch.exp(log_prob1) + self.pi2 * torch.exp(log_prob2))

In [ ]:
def plt_potential_func(log_prob, ax, num_grid=500, xs=[-5, 5], ys=[-5, 5], cmap=None):

    xlist = torch.linspace(xs[0], xs[1], num_grid)
    ylist = torch.linspace(ys[0], ys[1], num_grid)
    xx, yy = np.meshgrid(xlist, ylist)
    z = np.hstack([xx.reshape(-1, 1), yy.reshape(-1, 1)])

    z = torch.Tensor(z)
    u = log_prob(z).cpu().numpy()
    p = u.reshape(num_grid, num_grid)
    return ax.pcolor(xx, yy, p, cmap=cmap, vmin=-7.5, vmax=1.0)

gaussian_dist = Gaussian2(sigma=1.0)
fig, ax = plt.subplots(figsize=(4, 4), frameon=False)
im = plt_potential_func(gaussian_dist.log_prob, ax)
ax.axis('equal')
ax.axis("off")
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
plt.margins(0,0)
display(fig)
plt.close()
fig.savefig('gaussian2_true.png', pad_inches = 0,transparent = False)

In [ ]:
# define the gradient calculation
def grad_logp(x, logp):
    return torch.autograd.grad(logp(x), x, grad_outputs=torch.ones_like(logp(x)),
                               create_graph=False)[0]

sigmas = [0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
scatters = []
titles = []
all_flows = []

for i, sigma in enumerate(sigmas):
    gaussian_dist = Gaussian2(sigma=sigma)

    # initialize 1,000 particles
    xs_lang = torch.randn([1024, 2])

    scatter_plot = axes[i].scatter(xs_lang[:, 0], xs_lang[:, 1], s=5, alpha=0.2, color='red')
    time_text = axes[i].text(1.0, -3.5, '$t=0$', fontsize=14)
    scatters.append(scatter_plot)
    titles.append(time_text)
    axes[i].axis('equal')
    axes[i].set_xlim(-5, 5)
    axes[i].set_ylim(-5, 5)
    axes[i].axis("off") # remove the axis will speed up the training
    fig.tight_layout()

    clear_output(wait=True)
    display(fig)
    xs_lang.requires_grad=True
    flows = [xs_lang.detach()]

    n_iters = 1500
    lr = 1e-5
    decay_rate = 0.999
    # Unadjusted Langevin dynamics step
    for i in range(n_iters+1):
        xs_lang = xs_lang + 0.001*grad_logp(xs_lang, gaussian_dist.log_prob) + np.sqrt(0.001*2) * torch.randn_like(xs_lang)
        lr = lr * decay_rate
        flows.append(xs_lang.detach())
        scatter_plot.set_offsets(xs_lang.detach())
        time_text.set_text(f'$t={i}/{n_iters}$')
        clear_output(wait=True)
        display(fig)
        print('[%d/%d]: lr, %.7f' % (i, n_iters, lr))
    plt.close()

    all_flows.append(flows)


In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter
from tqdm import tqdm

def update(frame):
    for i in range(len(sigmas)):
        scatters[i].set_offsets(all_flows[i][frame])
        titles[i].set_text(f'$\sigma={sigmas[i]}, t={frame}/{n_iters}$')
## Generate gif
ani = FuncAnimation(fig=fig, func=update, frames=tqdm(range(0, len(flows), 1)), interval=1)
writer = PillowWriter(fps=40)
ani.save("gaussian2_langevin.gif", writer=writer)